In [1]:
import random

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from google.colab import userdata
from sklearn import metrics
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms


In [2]:
seed = 100
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [3]:
def sim_auc(similarities, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        tuple: (average_auc, average_fpr95)
    """
    if len(similarities) != len(datasets):
        raise ValueError("Number of similarities arrays must match number of dataset names")

    if len(similarities) < 2:
        raise ValueError("At least 2 datasets (ID and OOD) are required")

    similarities = np.array(similarities, dtype=object)  # Use object dtype for arrays of different lengths
    id_confi = similarities[0]

    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f"Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}")

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print("-" * 60)
    print(f"Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}")

    return avg_auc, avg_fpr


def sim_ap(similarities, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of similarity arrays, first one is ID dataset
        datasets (list): List of dataset names

    Returns:
        float: average AP score
    """
    if len(similarities) != len(datasets):
        raise ValueError("Number of similarities arrays must match number of dataset names")

    if len(similarities) < 2:
        raise ValueError("At least 2 datasets (ID and OOD) are required")

    similarities = np.array(similarities, dtype=object)
    id_confi = similarities[0]

    ap_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f"Dataset: {dataset:<25} | AP: {aver_p:.4f}")

    avg_ap = np.mean(ap_scores)
    print("-" * 40)
    print(f"Average AP: {avg_ap:.4f}")

    return avg_ap


def calculate_auc_metrics(id_conf, ood_conf):
    """
    Calculate AUROC and FPR at 95% TPR for binary classification.

    Args:
        id_conf (np.ndarray): Confidence scores for ID (in-distribution) samples
        ood_conf (np.ndarray): Confidence scores for OOD (out-of-distribution) samples

    Returns:
        tuple: (auroc, fpr_at_95_tpr)
    """
    # Combine predictions and create labels
    all_conf = np.concatenate([id_conf, ood_conf])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    # Calculate ROC curve
    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)

    # Calculate AUROC
    auroc = metrics.auc(fpr, tpr)

    # Calculate FPR at 95% TPR
    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        print(f"Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}")

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    # Combine predictions and create labels
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    # ID samples are positive (1), OOD samples are negative (0)
    labels = np.concatenate([np.ones(len(id_predictions)), np.zeros(len(ood_predictions))])

    # Calculate Average Precision
    average_precision = metrics.average_precision_score(labels, all_predictions)

    return average_precision


In [4]:
DEFAULT_MEAN = (0.485, 0.456, 0.406)
DEFAULT_STD = (0.229, 0.224, 0.225)

# 1. Load the dataset from Hugging Face
HF_TOKEN = userdata.get('HF_TOKEN')
dataset = load_dataset('TheKernel01/MS-COCOAI', token=HF_TOKEN)

data_split = dataset['test']

README.md: 0.00B [00:00, ?B/s]

data/validation-00000-of-00002.parquet:   0%|          | 0.00/333M [00:00<?, ?B/s]

data/validation-00001-of-00002.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/448M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/445M [00:00<?, ?B/s]

...

data/test-00006-of-00008.parquet:   0%|          | 0.00/443M [00:00<?, ?B/s]

data/test-00007-of-00008.parquet:   0%|          | 0.00/449M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/42000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/45000 [00:00<?, ? examples/s]

In [5]:
# 2. Create a custom PyTorch Dataset wrapper for the Hugging Face dataset
class HFImageDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data)

    def __getitem__(self, idx):
        item = self.hf_data[idx]

        # Ensure the image is in 3-channel RGB format
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [6]:
class RIGID_Detector():

    def __init__(self, lamb=0.05, percentile=5):
        self.lamb = lamb
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').cuda()
        self.model.eval()

    @torch.no_grad()
    def calculate_sim(self, data):
        features = self.model(data)
        noise = torch.randn_like(data).to(data.device)
        trans_data = data + noise * self.lamb
        trans_features = self.model(trans_data)
        sim_feat = F.cosine_similarity(features, trans_features, dim=-1)
        return sim_feat

    @torch.no_grad()
    def detect(self, data):
        sim = self.calculate_sim(data)
        return sim

In [7]:
transform_RIGID = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=DEFAULT_MEAN, std=DEFAULT_STD),
])

In [ ]:
# 3. Filter the dataset into Real (ID) and specific AI-generated (OOD) subsets
# Define the mapping based on the dataset documentation
generator_mapping = {
    1: 'SD21',
    2: 'SDXL',
    3: 'SD3',
    4: 'DALLE3',
    5: 'Midjourney'
}

# 3a. Extract the Real (ID) dataset first (Generator = 0)
real_hf = data_split.filter(lambda x: x['generator'] == 0)
real_dataset = HFImageDataset(real_hf, transform=transform_RIGID)

evaluation_datasets = [
    ('Real (ID)', real_dataset)
]

# 3b. Extract each Fake (OOD) dataset by its specific generator ID
for gen_id, gen_name in generator_mapping.items():
    fake_hf = data_split.filter(lambda x: x['generator'] == gen_id)
    fake_dataset = HFImageDataset(fake_hf, transform=transform_RIGID)
    evaluation_datasets.append((f'{gen_name} (OOD)', fake_dataset))

test_datasets = [name for name, _ in evaluation_datasets]
noise_intensity = 0.05
batch_size = 256

rigid_detector = RIGID_Detector(lamb=noise_intensity)

Filter:   0%|          | 0/45000 [00:00<?, ? examples/s]

In [ ]:
with torch.no_grad():
    sim_datasets = []

    for dataset_name, dataset_obj in evaluation_datasets:

        data_loader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True, num_workers=2)

        sim_feat = []
        total_num = 0

        for i, (samples, _) in enumerate(data_loader):

            samples = samples.cuda()
            samples_num = len(samples)
            total_num += samples_num

            sim = rigid_detector.calculate_sim(samples)
            sim_feat.append(sim)

            if total_num >= 1000: break

        if len(sim_feat) > 0:
            sim_feat = torch.cat(sim_feat, dim=0)
            print(f'{dataset_name:<25}, Image number: {sim_feat.shape[0]}, similarity is {sim_feat.mean().item():.4f}')
            sim_datasets.append(sim_feat.cpu().numpy())
        else:
            print(f'Warning: {dataset_name} is empty. Check your label filtering!')

    print("\n" + "=" * 60)
    print("Detection Results AUC (Per Generator):")
    print("=" * 60)
    sim_auc(sim_datasets, test_datasets)

    print("\n" + "=" * 60)
    print("Detection Results AP (Per Generator):")
    print("=" * 60)
    sim_ap(sim_datasets, test_datasets)

real_dataset = HFImageDataset(real_hf, transform=transform_RIGID)
fake_dataset = HFImageDataset(fake_hf, transform=transform_RIGID)

evaluation_datasets = [
    ('MS-COCO Real (ID)', real_dataset),
    ('MS-COCO AI-Generated (OOD)', fake_dataset)
]

test_datasets = [name for name, _ in evaluation_datasets]
noise_intensity = 0.05
batch_size = 256

rigid_detector = RIGID_Detector(lamb=noise_intensity)